In [38]:
import pandas as pd
import numpy as np
import joblib

In [39]:
model = joblib.load(
    "models/logistic_model.pkl"
)

In [40]:
current_claims = pd.read_csv(
    "Data/current_claims.csv"
)

In [41]:
from utils import engineer_features

In [42]:
current_claims = engineer_features(
    current_claims
)
print(current_claims)

       claim_id payer_id          payer_type   visit_type  total_billed  \
0    CCLM-00001     P004          Commercial    Emergency       2617.41   
1    CCLM-00002     P003          Commercial   Outpatient       3060.56   
2    CCLM-00003     P006  Medicare Advantage   Outpatient      11345.61   
3    CCLM-00004     P003          Commercial    Inpatient      47634.66   
4    CCLM-00005     P008        Medicaid MCO   Outpatient       2453.93   
..          ...      ...                 ...          ...           ...   
495  CCLM-00496     P005  Medicare Advantage  Observation       3869.97   
496  CCLM-00497     P001          Commercial  Observation       3740.88   
497  CCLM-00498     P007        Medicaid MCO    Emergency       9144.16   
498  CCLM-00499     P011          Commercial    Emergency      14296.33   
499  CCLM-00500     P011          Commercial    Inpatient      31619.27   

     expected_payment  num_procedures  num_diagnoses  prior_auth_required  \
0             1120.76 

In [43]:
denial_probability = model.predict_proba(
    current_claims.drop(
        columns=["claim_id","service_month"]
    )
)[:,1]

print(denial_probability)

[0.08473869 0.17029168 0.30325108 0.20951986 0.21777973 0.30021721
 0.15782129 0.12864363 0.22802743 0.06723025 0.34615318 0.16715237
 0.62229529 0.19805412 0.18071134 0.11902268 0.25992378 0.16121822
 0.34823449 0.16584845 0.11698912 0.04992297 0.65771525 0.13994709
 0.16623349 0.07495112 0.08892925 0.58767322 0.06769222 0.12889405
 0.14069169 0.28594167 0.07883343 0.06665966 0.13222979 0.19076955
 0.43500557 0.11350091 0.22402556 0.21937167 0.07222285 0.37777587
 0.10174281 0.27267409 0.19186641 0.27809485 0.10658079 0.17228988
 0.71006137 0.17393988 0.44884804 0.14323166 0.27416131 0.28407172
 0.33380495 0.0767246  0.10941243 0.35536457 0.10771082 0.12082792
 0.18643186 0.63362712 0.52300405 0.09490295 0.19774764 0.11459273
 0.15328411 0.37717748 0.11493716 0.17955384 0.11050254 0.08018056
 0.20669943 0.13839162 0.13249728 0.16066622 0.25506309 0.08202261
 0.12490742 0.21010594 0.15465063 0.73897627 0.10304252 0.07886654
 0.37877888 0.12886292 0.7910566  0.29709803 0.1495228  0.0501

In [44]:
current_claims["denial_probability"] = denial_probability

In [45]:
threshold = current_claims["denial_probability"].quantile(0.75)
print(threshold)

current_claims["predicted_denial"] = (
    current_claims["denial_probability"] >= threshold
).astype(int)

0.27379419122508586


In [46]:
current_claims["risk_tier"] = pd.qcut(
    current_claims["denial_probability"],
    q=[0,0.5,0.75,1],
    labels=["Low","Medium","High"]
)

In [47]:
current_claims = current_claims.sort_values(
    by="denial_probability",
    ascending=False
).reset_index(drop=True)

In [48]:
import shap

current_claims_copy = pd.read_csv(
    "Data/current_claims.csv"
)

current_claims_copy = engineer_features(
    current_claims_copy
)
print(current_claims_copy)

       claim_id payer_id          payer_type   visit_type  total_billed  \
0    CCLM-00001     P004          Commercial    Emergency       2617.41   
1    CCLM-00002     P003          Commercial   Outpatient       3060.56   
2    CCLM-00003     P006  Medicare Advantage   Outpatient      11345.61   
3    CCLM-00004     P003          Commercial    Inpatient      47634.66   
4    CCLM-00005     P008        Medicaid MCO   Outpatient       2453.93   
..          ...      ...                 ...          ...           ...   
495  CCLM-00496     P005  Medicare Advantage  Observation       3869.97   
496  CCLM-00497     P001          Commercial  Observation       3740.88   
497  CCLM-00498     P007        Medicaid MCO    Emergency       9144.16   
498  CCLM-00499     P011          Commercial    Emergency      14296.33   
499  CCLM-00500     P011          Commercial    Inpatient      31619.27   

     expected_payment  num_procedures  num_diagnoses  prior_auth_required  \
0             1120.76 

In [49]:
# Prepare model input (exclude non-feature columns)

X_current = current_claims_copy.drop(
    columns=["claim_id", "service_month"]
)

# Apply the same preprocessing used during training

X_current_processed = model.named_steps[
    "preprocessor"
].transform(X_current)

# Get transformed feature names

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

In [50]:
# Create SHAP explainer for Logistic Regression

explainer = shap.LinearExplainer(
    model.named_steps["classifier"],
    X_current_processed
)

# Calculate SHAP values

shap_values = explainer.shap_values(
    X_current_processed
)

# Convert to DataFrame

shap_df = pd.DataFrame(
    shap_values,
    columns=feature_names
)

Background dataset has 500 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=500 when initializing the masker.


In [51]:
def clean_feature_name(feature):

    feature = feature.replace("num__", "")
    feature = feature.replace("cat__", "")

    feature = feature.replace("_", " ")

    return feature.title()


def get_top_risk_factors(shap_row):

    # Top 3 features with largest contribution
    top_features = (
        shap_row.abs()
                .sort_values(ascending=False)
                .head(3)
                .index
    )

    return ", ".join(
        clean_feature_name(f)
        for f in top_features
    )

In [52]:
current_claims["top_risk_factors"] = shap_df.apply(
    get_top_risk_factors,
    axis=1
)

In [53]:
'''

def get_top_risk_factors(row):

    factors = []

    if row["auth_missing"] == 1:
        factors.append("Missing prior authorization")

    if row["missing_documentation_flag"] == 1:
        factors.append("Missing documentation")

    if row["eligibility_verified"] == 0:
        factors.append("Eligibility not verified")

    if row["referral_missing"] == 1:
        factors.append("Missing referral")

    if row["late_submission"] == 1:
        factors.append("Late claim submission")

    if row["is_in_network"] == 0:
        factors.append("Out-of-network provider")

    return ", ".join(factors[:3])

    '''

'\n\ndef get_top_risk_factors(row):\n\n    factors = []\n\n    if row["auth_missing"] == 1:\n        factors.append("Missing prior authorization")\n\n    if row["missing_documentation_flag"] == 1:\n        factors.append("Missing documentation")\n\n    if row["eligibility_verified"] == 0:\n        factors.append("Eligibility not verified")\n\n    if row["referral_missing"] == 1:\n        factors.append("Missing referral")\n\n    if row["late_submission"] == 1:\n        factors.append("Late claim submission")\n\n    if row["is_in_network"] == 0:\n        factors.append("Out-of-network provider")\n\n    return ", ".join(factors[:3])\n\n    '

In [54]:

print(current_claims["top_risk_factors"])

0      Auth Missing, Payer Type Commercial, Missing D...
1           Late Submission, Payer Id P003, Auth Missing
2      Eligibility Verified, Auth Missing, Payer Id P006
3      Payer Id P003, Auth Missing, Payer Type Commer...
4      Payer Id P008, Auth Missing, Missing Documenta...
                             ...                        
495    Auth Missing, Missing Documentation Flag, Paye...
496    Eligibility Verified, Payer Id P001, Auth Missing
497    Auth Missing, Missing Documentation Flag, Paye...
498    Payer Id P011, Auth Missing, Payer Type Commer...
499    Missing Documentation Flag, Late Submission, D...
Name: top_risk_factors, Length: 500, dtype: str


In [55]:
current_claims["explanation"] = ""

In [56]:
submission = current_claims[
    [
        "claim_id",
        "denial_probability",
        "predicted_denial",
        "risk_tier",
        "top_risk_factors",
        "explanation"
    ]
]

In [57]:
submission.to_csv(
    "predictions_current_claims.csv",
    index=False
)

In [58]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [59]:
load_dotenv(override=True)

# Create OpenAI client
# nice simple wrapper around making an HTTP call to an API cloud., it looks for gemini key by default,

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key = os.getenv("GOOGLE_API_KEY")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
MODEL="gemini-3.1-flash-lite"


In [75]:
print(current_claims.columns.tolist())

['claim_id', 'payer_id', 'payer_type', 'visit_type', 'total_billed', 'expected_payment', 'num_procedures', 'num_diagnoses', 'prior_auth_required', 'has_prior_auth', 'is_in_network', 'days_to_submit', 'missing_documentation_flag', 'eligibility_verified', 'referral_required', 'referral_present', 'service_month', 'payment_ratio', 'payment_gap', 'auth_missing', 'referral_missing', 'late_submission', 'service_month_num', 'denial_probability', 'predicted_denial', 'risk_tier', 'top_risk_factors', 'explanation']


In [86]:
def create_prompt(row):

    return f"""
You are an experienced healthcare claims analyst.

A machine learning model has predicted the denial risk for the following insurance claim.

Your responsibility is to explain the prediction using ONLY the information provided below.

==========================================================
CLAIM INFORMATION
==========================================================

Claim ID: {row['claim_id']}

Payer ID: {row['payer_id']}
Payer Type: {row['payer_type']}
Visit Type: {row['visit_type']}

Total Billed Amount: ${row['total_billed']:.2f}
Expected Payment: ${row['expected_payment']:.2f}

Number of Procedures: {row['num_procedures']}
Number of Diagnoses: {row['num_diagnoses']}

Prior Authorization Required: {"Yes" if row['prior_auth_required'] else "No"}
Prior Authorization Available: {"Yes" if row['has_prior_auth'] else "No"}

In Network Provider: {"Yes" if row['is_in_network'] else "No"}

Days to Submit: {row['days_to_submit']}

Missing Documentation: {"Yes" if row['missing_documentation_flag'] else "No"}

Eligibility Verified: {"Yes" if row['eligibility_verified'] else "No"}

Referral Required: {"Yes" if row['referral_required'] else "No"}
Referral Available: {"Yes" if row['referral_present'] else "No"}

Service Month: {row['service_month_num']}

Payment Ratio: {row['payment_ratio']:.3f}
Payment Gap: ${row['payment_gap']:.2f}

Authorization Missing Feature: {"Yes" if row['auth_missing'] else "No"}

Referral Missing Feature: {"Yes" if row['referral_missing'] else "No"}

Late Submission Feature: {"Yes" if row['late_submission'] else "No"}

==========================================================
MODEL OUTPUT
==========================================================

Denial Probability: {row['denial_probability']:.2%}

Predicted Denial: {"Yes" if row['predicted_denial'] else "No"}

Risk Tier: {row['risk_tier']}

Top ML Risk Factors:
{row['top_risk_factors']}

==========================================================
TASK
==========================================================

Use ONLY the information provided above.

Do NOT assume or invent any facts.

Focus primarily on the Top ML Risk Factors while using the remaining claim attributes as supporting evidence.

Write exactly 2–3 sentences that:

1. Explain why this claim has the predicted denial risk.
2. Mention the most influential contributing factors.
3. Recommend ONE practical action that could reduce denial risk before submission.
4. End with the following sentence exactly:

"This prediction represents estimated risk and does not guarantee claim denial."

Return ONLY the explanation.
"""

In [87]:

top10_claims = current_claims.head(10).copy()

In [88]:
print(create_prompt(top10_claims.iloc[0]))


You are an experienced healthcare claims analyst.

A machine learning model has predicted the denial risk for the following insurance claim.

Your responsibility is to explain the prediction using ONLY the information provided below.

CLAIM INFORMATION

Claim ID: CCLM-00087

Payer ID: P010
Payer Type: BCBS
Visit Type: Inpatient

Total Billed Amount: $30740.55
Expected Payment: $17759.27

Number of Procedures: 8
Number of Diagnoses: 11

Prior Authorization Required: No
Prior Authorization Available: Yes

In Network Provider: No

Days to Submit: 40

Missing Documentation: Yes

Eligibility Verified: No

Referral Required: No
Referral Available: No

Service Month: 1

Payment Ratio: 0.578
Payment Gap: $12981.28

Authorization Missing Feature: No

Referral Missing Feature: No

Late Submission Feature: Yes

MODEL OUTPUT

Denial Probability: 79.11%

Predicted Denial: Yes

Risk Tier: High

Top ML Risk Factors:
Auth Missing, Payer Type Commercial, Missing Documentation Flag

TASK

Use ONLY the 

In [89]:
response = gemini.chat.completions.create(

    model=MODEL,

    messages=[
        {
            "role": "user",
            "content": create_prompt(top10_claims.iloc[0])
        }
    ]
)

In [90]:
print(response.choices[0].message.content)

The 79.11% denial risk is primarily driven by missing documentation, the out-of-network provider status, and a late submission occurring 40 days post-service. To mitigate this high-risk status, the facility should immediately upload the required supporting medical documentation to rectify the missing information flag before the claim is processed. This prediction represents estimated risk and does not guarantee claim denial.


In [91]:
explanations = []

for _, row in top10_claims.iterrows():

    response = gemini.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "user",
                "content": create_prompt(row)
            }
        ]

    )

    explanations.append(
        response.choices[0].message.content
    )

In [92]:
top10_claims["explanation"] = explanations

In [93]:
submission.loc[
    submission.index[:10],
    "explanation"
] = top10_claims["explanation"].values

In [94]:
submission.to_csv(
    "predictions_current_claims.csv",
    index=False
)

#### Low Risk LLM response

In [97]:
low_risk_claim = current_claims[
    current_claims["risk_tier"] == "Low"
].iloc[0]

print(low_risk_claim)

claim_id                                                             CCLM-00452
payer_id                                                                   P007
payer_type                                                         Medicaid MCO
visit_type                                                          Observation
total_billed                                                            9736.08
expected_payment                                                        3451.45
num_procedures                                                                4
num_diagnoses                                                                 3
prior_auth_required                                                           0
has_prior_auth                                                                0
is_in_network                                                                 0
days_to_submit                                                               24
missing_documentation_flag              

In [96]:
response = gemini.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": create_prompt(low_risk_claim)
        }
    ]
)

print(response.choices[0].message.content)

Although the model identifies factors such as potential authorization gaps and missing documentation as theoretical risks, the claim's low denial probability of 16.59% is supported by the fact that the actual claim status indicates no documentation is missing and no authorization is required. Given that the provider is out-of-network, the primary recommendation to further secure this claim is to verify that the specific Medicaid MCO contract allows for out-of-network reimbursement for observation services. This prediction represents estimated risk and does not guarantee claim denial.
